### Middleware
Middleware provides a way to more tightly control what happens inside the agent. Middleware is useful for the following:

• Tracking agent behavior with logging, analytics, and debugging.

• Transforming prompts, tool selection, and output formatting.

• Adding retries, fallbacks, and early termination logic.

• Applying rate limits, guardrails, and PII detection.

In [1]:
import os
from dotenv import load_dotenv
load_dotenv()

os.environ["GOOGLE_API_KEY"] = os.getenv("GOOGLE_API_KEY")


### Summarization MiddleWare

Automatically summarize conversation history when approaching token limits, preserving recent messages while compressing older context. Summarization is useful for the following:

Long-running conversations that exceed context windows.

Multi-turn dialogues with extensive history.

Applications where preserving full conversation context matters.


In [3]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.messages import HumanMessage, SystemMessage

### Message based summarization
agent = create_agent(
    model="google_genai:gemini-3.5-flash-lite",
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model="google_genai:gemini-3.5-flash-lite",     ##use model that consumes less tokens
            trigger=("messages", 10),
            keep=("messages", 4)
        )
    ]
)


In [4]:
### Run with thread id
config={"configureable":{"thread_id":"test-1"}}

In [6]:
### Run with thred id
config={"configureable":{"thread_id":"test-1"}}

In [10]:
config = {"configurable": {"thread_id": "1"}}

questions = [
    "what is 2+2?",
    "what is 10*5?",
    "what is 2/22?",
    "what is 215-7?",
    "what is 3*3?",
    "what is 4*4?"
]

for q in questions:
    response = agent.invoke({"messages": [HumanMessage(content=q)]}, config)
    print(f"Total messages in state: {len(response['messages'])}")
    print(f"Latest reply: {response['messages'][-1].content}\n")


Total messages in state: 2
Latest reply: [{'type': 'text', 'text': '2 + 2 = 4', 'extras': {'signature': 'EnsKeQFpFH0TPs0pbKwtr2GnXnz5RwZ6RyIQ+k8r4d8f7u0ytpSKIqotJ25g1sWlg6rbayZWVPhDFUdU4k2nFbkTeAmTzVL9a90zG11QuCpWhFVnhdMNrQ6pUsEoiXnUt4RgEhAIL13SQl5LEUXSEwOplCWSXOgVDScbu3s='}}]

Total messages in state: 4
Latest reply: [{'type': 'text', 'text': '10 * 5 = 50', 'extras': {'signature': 'EnsKeQFpFH0TqrX8UMec0yPMKBRVRqC4+9m9a859Y/rHvPph1AsoeNMcCK+Y+pQv5Fj5KIIB7f0kGi4l2iJcHFBERcjHngZNjCJwtv2dpcPiNiNj9qI5CFamy4d2PqEpUEtdLHALGD2Cdh19ILcbpHt9erMiZ2zvEtrftYE='}}]

Total messages in state: 6
Latest reply: [{'type': 'text', 'text': '2 / 22 simplifies to **1/11**, which is approximately **0.0909** (or 0.090909...).', 'extras': {'signature': 'EnsKeQFpFH0TZ6MabHfC8dxfqp2hqSw4ecsZsV5NG1pEanBgzDcOLzHKk7i2Jjp44bb3tYCuKswkaVS9wOhN1ZZ+R0bKxRQjCyqSYv0QPxxciBwkRW79Yguf8Ge3ajto8IsOXp37oZVKi0yb1VyAVvIjXKmccJYfTGbwd+k='}}]

Total messages in state: 8
Latest reply: [{'type': 'text', 'text': '215 - 7 = 208', 'ext

### Token Size

In [11]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
from langgraph.checkpoint.memory import InMemorySaver

@tool
def search_hotels(city: str) -> str:
    """Search hotels - returns long response to use more tokens."""
    return f"""Hotels in {city}:
    1. Grand Hotel - 5 star, $350/night, spa, pool, gym
    2. City Inn - 4 star, $180/night, business center
    3. Budget Stay - 3 star, $75/night, free wifi"""

agent=create_agent(
    model="google_genai:gemini-3.5-flash-lite",     ##use model that consumes less tokens
    tools=[search_hotels],
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model="google_genai:gemini-3.5-flash-lite",     
            trigger=("tokens", 550),
            keep=("tokens", 200)
        )
    ]
)
config = {"configurable": {"thread_id": "1"}}
# Token counter (approximate)
def count_tokens(messages):
    total_chars = sum(len(str(m.content)) for m in messages)
    return total_chars // 4  # 4 chars ≈ 1 token


In [12]:
# Run test
cities = ["Islamabad", "Lahore", "Rawalpindi", "New York", "Dubai", "Singapore"]

for city in cities:
    response = agent.invoke(
        {"messages": [HumanMessage(content=f"Find hotels in {city}")]},
        config=config
    )

    tokens = count_tokens(response["messages"])
    print(f"{city}: ~{tokens} tokens, {len(response['messages'])} messages")
    print(f"{(response['messages'])}")


Islamabad: ~207 tokens, 4 messages
[HumanMessage(content='Find hotels in Islamabad', additional_kwargs={}, response_metadata={}, id='56bbaff4-2548-4a13-8869-e25cddc17163'), AIMessage(content=[], additional_kwargs={'function_call': {'name': 'search_hotels', 'arguments': '{"city": "Islamabad"}'}, '__gemini_function_call_thought_signatures__': {'call_3423426678440607759': 'EnsKeQFpFH0T/UpZb/4GmIiVfvGG3npqadtT0j3/i6rHPNgoBqOReAT3bkBSKkLIy61D8wTWsFFn5dI8Z8StAo6IaF1uX/FiQCYfNNc9f7pf7Kl8xlruJkZdTcsImc2AtN5wblVWwG9yr5Q+8ykB9whmwfZZjF3Hg2FLYpk='}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.5-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--01a1f8bc-21bb-7176-a481-5bc61a20b106-0', tool_calls=[{'name': 'search_hotels', 'args': {'city': 'Islamabad'}, 'id': 'call_3423426678440607759', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 53, 'output_tokens': 17, 'total_tokens': 70, 'input_token_details': {'cache_read

### Fractions

In [15]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
from langgraph.checkpoint.memory import InMemorySaver

@tool
def search_hotels(city: str) -> str:
    """Search hotels - returns long response to use more tokens."""
    return f"""Hotels in {city}:
    1. Grand Hotel - 5 star, $350/night, spa, pool, gym
    2. City Inn - 4 star, $180/night, business center
    3. Budget Stay - 3 star, $75/night, free wifi"""

# Token counter (approximate)
def count_tokens(messages):
    total_chars = sum(len(str(m.content)) for m in messages)
    return total_chars // 4  # 4 chars ≈ 1 token

# Fraction-based summarization agent
agent = create_agent(
    model="google_genai:gemini-3.5-flash-lite",
    tools=[search_hotels],
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model="google_genai:gemini-3.5-flash-lite",
            token_counter=count_tokens,
            trigger=("fraction", 0.8),  # Trigger when 80% of max_input_tokens is reached
            keep=("fraction", 0.3),     # Keep the most recent 30% of max_input_tokens
        )
    ]
)

# Run test
config = {"configurable": {"thread_id": "1"}}
cities = ["Paris", "London", "Tokyo", "New York", "Dubai", "Singapore"]

for city in cities:
    response = agent.invoke(
        {"messages": [HumanMessage(content=f"Find hotels in {city}")]},
        config=config
    )

    tokens = count_tokens(response["messages"])
    print(f"{city}: ~{tokens} tokens, {len(response['messages'])} messages")


Paris: ~163 tokens, 4 messages
London: ~327 tokens, 8 messages
Tokyo: ~491 tokens, 12 messages
New York: ~664 tokens, 16 messages
Dubai: ~828 tokens, 20 messages
Singapore: ~994 tokens, 24 messages


# LangChain Agent Core Concepts

## 1. Middleware
**Middleware** is a control layer that sits between your application and the AI model to monitor, modify, or manage what happens inside an agent loop.
* **Purpose:** Adds guardrails, logging, retries, memory compression, or human approval steps without changing your core model logic.
* **Example (`SummarizationMiddleware`):** Watches the conversation length and automatically summarizes older messages before the model hits its memory limit.

---

## 2. Messages
**Messages** are the fundamental units of conversation state passed between the user, the AI model, and external tools.
* **`SystemMessage`:** Sets the AI's persona, tone, and behavioral rules at the beginning of a chat.
* **`HumanMessage`:** Represents input or questions sent by the user.
* **`AIMessage`:** Represents replies generated by the model or structured requests (`tool_calls`) to execute a function.
* **`ToolMessage`:** Carries the output returned by an executed tool back to the model (linked via `tool_call_id`).

---

## 3. Tokens
**Tokens** are the small chunks of text (words, sub-words, or characters) that language models read, process, and count to measure text size.
* **Rule of Thumb:** In English, **1 token $\approx$ 4 characters** (or about $\frac{3}{4}$ of a word).
* **Context Window:** Every model has a maximum token limit per conversation. Tracking tokens prevents the conversation history from exceeding this limit and causing errors.

---

## 4. Fractions
**Fractions** are decimal percentages (`0.0` to `1.0`) used in `SummarizationMiddleware` to dynamically manage memory based on a model's maximum token limit.
* **`trigger=("fraction", 0.8)`:** Automatically triggers summarization when the conversation reaches **80%** of the model's context window.
* **`keep=("fraction", 0.3)`:** Preserves the most recent **30%** of the context window untouched after summarizing older messages.
* **Benefit:** Scales automatically across models with different context window sizes without hardcoding token counts.

---

## 5. Human-in-the-Loop (HITL) Middleware
**Human-in-the-Loop Middleware (`HumanInTheLoopMiddleware`)** pauses agent execution right before calling sensitive or high-stakes tools so a human can review the action.
* **Requirement:** Must be paired with a checkpointer (such as `InMemorySaver`) and a `thread_id` to save and resume paused states.
* **Allowed Decisions (`Command(resume=...)`):**
  1. **`approve`:** Runs the tool as requested by the model.
  2. **`edit`:** Modifies the tool's arguments before execution.
  3. **`reject`:** Cancels the tool call and sends feedback back to the agent.


In [16]:
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver

def read_email_tool(email_id: str) -> str:
    """Mock function to read an email by its ID."""
    return f"Email content for ID: {email_id}"

def send_email_tool(recipient: str, subject: str, body: str) -> str:
    """Mock function to send an email."""
    return f"Email sent to {recipient} with subject '{subject}'"


In [17]:
agent = create_agent(
    model="google_genai:gemini-3.5-flash-lite",
    tools=[read_email_tool, send_email_tool],
    checkpointer=InMemorySaver(),
    middleware=[
       HumanInTheLoopMiddleware(
            interrupt_on={
                # Pause before calling send_email; allow approve, edit, or reject
                "send_email_tool": {
                    "allowed_decisions": ["approve", "edit", "reject"]
                },
                "read_email_tool":False
            }
        )
    ]
)

In [18]:
config = {"configurable": {"thread_id": "test-approve"}}
# Step 1: Request
result = agent.invoke(
    {"messages": [HumanMessage(content="Send email to john@test.com with subject 'Hello' and body 'How are you?'")]},
    config=config
)


In [19]:
result

{'messages': [HumanMessage(content="Send email to john@test.com with subject 'Hello' and body 'How are you?'", additional_kwargs={}, response_metadata={}, id='bef44b41-9831-42db-a1b5-f73ad3506b91'),
  AIMessage(content=[], additional_kwargs={'function_call': {'name': 'send_email_tool', 'arguments': '{"subject": "Hello", "recipient": "john@test.com", "body": "How are you?"}'}, '__gemini_function_call_thought_signatures__': {'call_12042253195302599363': 'Eo4BCosBAWkUfROM7j7HxZJ3E50qKUyUT9jHf7h2/4WwnE9uVvue4/2CF5sUKE4Iw3gmDvCG1DhwHq4w5ZEZQ3Y1DvvS9YuRtZLAtoD03m3BpZWeDzQMsWu5l+bU+THn3xyjF/qlX9vMJP+Ap99lKJlPAVZPJIm+B/I5YWofZIYFRhHVDu8FRnQAUIBrPjdqfg=='}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.5-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--01a1f8cc-956c-76a8-8799-064ed170390d-0', tool_calls=[{'name': 'send_email_tool', 'args': {'subject': 'Hello', 'recipient': 'john@test.com', 'body': 'How are you?'}, 'id': 'call_1204225319530

In [21]:
from langgraph.types import Command
# Step 2: Approve
if "__interrupt__" in result:
    print("⏸️ Paused! Approving...")

    result = agent.invoke(
        Command(
            resume={
                "decisions": [
                    {"type": "approve"}
                ]
            }
        ),
        config=config
    )

    print(f"✅ Result: {result['messages'][-1].content}")


⏸️ Paused! Approving...
✅ Result: [{'type': 'text', 'text': 'Email has been successfully sent to john@test.com with the subject "Hello".', 'extras': {'signature': 'EnsKeQFpFH0TpqH3YIztTfpFCofogvm2DAnIXfarsHgTxfYFUKSEUUMVqutVLGzmcUIREq4/z8wBLq+hn+E73FHJKVCqdFRlA3F+tw/B1Vi40hKBEQuS/OFBywsmViT8NhgJbwuIr6kFjOo1rTmfmPEd2GjrL3XG0/lg0VY='}}]


In [22]:
result

{'messages': [HumanMessage(content="Send email to john@test.com with subject 'Hello' and body 'How are you?'", additional_kwargs={}, response_metadata={}, id='bef44b41-9831-42db-a1b5-f73ad3506b91'),
  AIMessage(content=[], additional_kwargs={'function_call': {'name': 'send_email_tool', 'arguments': '{"subject": "Hello", "recipient": "john@test.com", "body": "How are you?"}'}, '__gemini_function_call_thought_signatures__': {'call_12042253195302599363': 'Eo4BCosBAWkUfROM7j7HxZJ3E50qKUyUT9jHf7h2/4WwnE9uVvue4/2CF5sUKE4Iw3gmDvCG1DhwHq4w5ZEZQ3Y1DvvS9YuRtZLAtoD03m3BpZWeDzQMsWu5l+bU+THn3xyjF/qlX9vMJP+Ap99lKJlPAVZPJIm+B/I5YWofZIYFRhHVDu8FRnQAUIBrPjdqfg=='}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.5-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--01a1f8cc-956c-76a8-8799-064ed170390d-0', tool_calls=[{'name': 'send_email_tool', 'args': {'subject': 'Hello', 'recipient': 'john@test.com', 'body': 'How are you?'}, 'id': 'call_1204225319530

### Reject

In [23]:
config = {"configurable": {"thread_id": "test-reject"}}
# Step 1: Request
result = agent.invoke(
    {"messages": [HumanMessage(content="Send email to john@test.com with subject 'Hello' and body 'How are you?'")]},
    config=config)


In [24]:
from langgraph.types import Command

# Step 2: Approve
if "__interrupt__" in result:
    print("⏸️ Paused! Approving...")

    result = agent.invoke(
        Command(
            resume={
                "decisions": [
                    {"type": "reject"}
                ]
            }
        ),
        config=config
    )

    print(f"✅ Result: {result['messages'][-1].content}")


⏸️ Paused! Approving...
✅ Result: [{'type': 'text', 'text': 'The email sending request was rejected and the tool was not executed. If you would like me to try again or need help with something else, please let me know.', 'extras': {'signature': 'EnsKeQFpFH0T5NibXVmpnBk4bVbk41byDU7/8CrfZ/ypmedl24wKDFtwk/dFiL1HiHUjdWVrIFkoISw3gQRNGCO//uZZJ6qLaM0zru9T7DOUnx+TeKwhcpZG48B4+VuM0mkmulmFs25jW4UzG+8BMhQ9ve2AZhBAyCr2wEw='}}]


### Editing

In [25]:
config = {"configurable": {"thread_id": "test-edit"}}

# Step 1: Request (with wrong info)
result = agent.invoke(
    {"messages": [HumanMessage(content="Send email to wrong@email.com with subject 'Test' and body 'Hello'")]},
    config=config
)


In [27]:
result

{'messages': [HumanMessage(content="Send email to wrong@email.com with subject 'Test' and body 'Hello'", additional_kwargs={}, response_metadata={}, id='b505c2e2-887f-4511-a3ba-286b2145b118'),
  AIMessage(content=[], additional_kwargs={'function_call': {'name': 'send_email_tool', 'arguments': '{"body": "Hello", "recipient": "wrong@email.com", "subject": "Test"}'}, '__gemini_function_call_thought_signatures__': {'call_7146516421801202235': 'EnsKeQFpFH0TUnqhBHownT3aVXQCdQoYEXvBEaJh6Plo2ERuvwOhRkEyTbG3beav//zN5yuBujGeZPEimmo/YczOVHqhYtkY6JFVIKGCVHLpSKOq9naHLWjFz5qss08vL8hmtMHbpcHg9EselkJeAF2Zwh1kkKZ76pQQ6q8='}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.5-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--01a1f8d0-e3f5-722f-8d16-a048bdb62d78-0', tool_calls=[{'name': 'send_email_tool', 'args': {'body': 'Hello', 'recipient': 'wrong@email.com', 'subject': 'Test'}, 'id': 'call_7146516421801202235', 'type': 'tool_call'}], invalid_tool_ca

In [26]:
# Step 2: Edit and approve
if "__interrupt__" in result:
    print("⏸️ Paused! Editing...")

    result = agent.invoke(
        Command(
            resume={
                "decisions": [
                    {
                        "type": "edit",
                        "edited_action": {
                            "name": "send_email_tool",      # Tool name
                            "args": {                       # New arguments
                                "recipient": "correct@email.com",
                                "subject": "Corrected Subject",
                                "body": "This was edited by human before sending"
                            }
                        }
                    }
                ]
            }
        ),
        config=config
    )


⏸️ Paused! Editing...


In [28]:
result

{'messages': [HumanMessage(content="Send email to wrong@email.com with subject 'Test' and body 'Hello'", additional_kwargs={}, response_metadata={}, id='b505c2e2-887f-4511-a3ba-286b2145b118'),
  AIMessage(content=[], additional_kwargs={'function_call': {'name': 'send_email_tool', 'arguments': '{"body": "Hello", "recipient": "wrong@email.com", "subject": "Test"}'}, '__gemini_function_call_thought_signatures__': {'call_7146516421801202235': 'EnsKeQFpFH0TUnqhBHownT3aVXQCdQoYEXvBEaJh6Plo2ERuvwOhRkEyTbG3beav//zN5yuBujGeZPEimmo/YczOVHqhYtkY6JFVIKGCVHLpSKOq9naHLWjFz5qss08vL8hmtMHbpcHg9EselkJeAF2Zwh1kkKZ76pQQ6q8='}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.5-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--01a1f8d0-e3f5-722f-8d16-a048bdb62d78-0', tool_calls=[{'name': 'send_email_tool', 'args': {'body': 'Hello', 'recipient': 'wrong@email.com', 'subject': 'Test'}, 'id': 'call_7146516421801202235', 'type': 'tool_call'}], invalid_tool_ca